In [1]:
import ibis
from ibis import _, selectors as s
import pandas as pd
from utils.f_0_dirs import get_data_dirs

# --- #
table_fixed_name = "fame_fixed_filtered"
table_panel_name = "fame_yearly_kp"

# --- #
dirs = get_data_dirs(segment="descriptives")
con = ibis.duckdb.connect(dirs.db_path)

t_fixed_mod = (
    con.table(table_fixed_name)
    .mutate(
        entity_string=_.entity_type.fillna("missing"),
        has_guo=_.guo.notnull().ifelse(1, 0),
        entity_dummy=_.entity_type.notnull().ifelse(1, 0),
        latest_accounts_years = _.latest_accounts_date.epoch_seconds() / 60 / 60 / 24 / 365.25 + 1970
    )
    .select([
        'registered_number',
        'latest_accounts_years',
        'has_guo', 'entity_string', 'entity_dummy'
    ])
    .pivot_wider(
        names_from="entity_string",
        values_from="entity_dummy",
        names_prefix='entity_type_'
    )
)
t_panel_mod = (
    con.table(table_panel_name)
    .mutate(
        consolidated_flag=_.consolidated.notnull().ifelse(1, 0),
    )
    .drop('consolidated')
)
t_panel_merged = (
    t_panel_mod
    .left_join(t_fixed_mod, ['registered_number'])
    .drop('registered_number_right')
)

index_cols = ['registered_number', 'year']
boolean_cols = ['consolidated_flag', 'has_guo'] + [col for col in t_panel_merged.columns if col.startswith('entity_type_')]
avoid_cols = index_cols + boolean_cols
t_scaled = (
    t_panel_merged
    .mutate(**{
        col: (_[col] - _[col].mean()) / _[col].std() 
        for col in t_panel_merged.columns if col not in avoid_cols
    })
)

print(f"Shape: {t_panel_merged.count().execute()}x{len(t_panel_merged.columns)}")
print(t_panel_merged.schema())
print(t_panel_merged.limit(5).execute())

# Aggregate stats
df_stats_wide = (
    t_scaled
    .drop(avoid_cols)
    .aggregate(s.across(s.all(), { "mean": _.mean().round(4), "std": _.std() }))
    .execute()
)
s_flat = df_stats_wide.iloc[0]
s_flat.index = pd.MultiIndex.from_tuples(
    [col.rsplit("_", 1) for col in s_flat.index], 
    names=["column", "stat"]
)
df_stats = s_flat.unstack(level="stat")[["mean", "std"]]
display(df_stats)

c:\Users\lazyst\Files\ucl\Dissertation\.venv-main\Lib\site-packages\ibis\common\deferred.py:414: FutureWarning: `Value.fillna` is deprecated as of v9.1; use fill_null instead
  return func(*args, **kwargs)


Shape: 1135773x45
ibis.Schema {
  registered_number              string
  year                           int64
  turnover                       float64
  shareholders_funds             float64
  profit_loss_pretax             float64
  employees                      int64
  tangibles                      float64
  tangibles_land_and_buildings   float64
  tangibles_land_freehold        float64
  tangibles_land_leasehold       float64
  tangibles_fixt_fit             float64
  tangibles_plant_and_vehicles   float64
  tangibles_plant                float64
  tangibles_vehicles             float64
  fixed_other                    float64
  intangibles                    float64
  investments_other              float64
  fixed_total                    float64
  liabilities                    float64
  total_assets                   float64
  liabilites_lt                  float64
  cos                            float64
  admin_expenses                 float64
  interest_paid               

stat,mean,std
column,,
admin_expenses,-0.0,1.0
cos,0.0,1.0
depreciation,-0.0,1.0
dividends,-0.0,1.0
ebitda,0.0,1.0
employees,-0.0,1.0
fixed_other,0.0,1.0
fixed_total,-0.0,1.0
intangibles,0.0,1.0


In [2]:
df_merged = t_scaled.execute()

In [3]:
from fancyimpute import SoftImpute

df_features = (
    df_merged
    .drop(columns=['registered_number', 'year'])
)
X = df_features.to_numpy(dtype=float)

# Count NaN values in the whole matrix
count_entries = df_features.size
count_nan = df_features.isna().sum().sum()
print(f"Total NaN values in the features DataFrame: {count_nan:,} / {count_entries:,} ({count_nan / count_entries:.1%})")

df_imputed = SoftImpute(max_iters=100).fit_transform(df_features)
count_i_entries = df_imputed.size
count_i_nan = pd.isna(df_imputed).sum().sum()
print(f"Total NaN values in the imputed features DataFrame: {count_i_nan:,} / {count_i_entries:,} ({count_i_nan / count_i_entries:.1%})")

Total NaN values in the features DataFrame: 17,923,251 / 48,838,239 (36.7%)
[SoftImpute] Max Singular Value of X_init = 3338.335090
[SoftImpute] Iter 1: observed MAE=0.011910 rank=31
[SoftImpute] Iter 2: observed MAE=0.011690 rank=30
[SoftImpute] Iter 3: observed MAE=0.011510 rank=29
[SoftImpute] Iter 4: observed MAE=0.011392 rank=29
[SoftImpute] Iter 5: observed MAE=0.011313 rank=29
[SoftImpute] Iter 6: observed MAE=0.011256 rank=29
[SoftImpute] Iter 7: observed MAE=0.011213 rank=29
[SoftImpute] Iter 8: observed MAE=0.011180 rank=29
[SoftImpute] Iter 9: observed MAE=0.011154 rank=29
[SoftImpute] Iter 10: observed MAE=0.011132 rank=29
[SoftImpute] Iter 11: observed MAE=0.011114 rank=29
[SoftImpute] Iter 12: observed MAE=0.011098 rank=29
[SoftImpute] Iter 13: observed MAE=0.011083 rank=29
[SoftImpute] Iter 14: observed MAE=0.011070 rank=29
[SoftImpute] Iter 15: observed MAE=0.011057 rank=29
[SoftImpute] Iter 16: observed MAE=0.011045 rank=29
[SoftImpute] Iter 17: observed MAE=0.011033 r

In [6]:
from numpy.linalg import matrix_rank

print(f"Rank of imputed features matrix: {matrix_rank(df_imputed)}")
print("Original matrix preview: ")
print(df_features.head())
print("Imputed matrix preview: ")
print(df_imputed[:5,])

Rank of imputed features matrix: 42
Original matrix preview: 
   turnover  shareholders_funds  profit_loss_pretax  employees  tangibles  \
0  0.083384            0.010006            0.007452   0.741957   0.040897   
1 -0.007825           -0.014382           -0.000154  -0.080087  -0.036997   
2 -0.009082           -0.015178            0.000022  -0.081742        NaN   
3  0.060551            0.026858            0.012361   1.300466   0.323743   
4 -0.008910           -0.015493           -0.000103  -0.086086        NaN   

   tangibles_land_and_buildings  tangibles_land_freehold  \
0                      0.025370                      NaN   
1                           NaN                      NaN   
2                           NaN                      NaN   
3                      0.181204                      NaN   
4                           NaN                      NaN   

   tangibles_land_leasehold  tangibles_fixt_fit  tangibles_plant_and_vehicles  \
0                       NaN      

In [ ]:
from sklearn.decomposition import PCA
from scipy.sparse.linalg import svds as svds
import pandas as pd
import numpy as np

n_components = 6

X = df_imputed
# pca = PCA(n_components=n_components)
# pc_scores = pca.fit_transform(df_features)

# Give me the type of every column in df_features

u6, s6, v6_T = svds(X, k=n_components)
# 4. Reverse arrays to order components by largest singular value first
idx = np.argsort(s6)[::-1]
s6 = s6[idx]
u6 = u6[:, idx]
v6_T = v6_T[idx, :]

# 5. Calculate Total Variance & Explained Variance
total_variance = np.sum(X ** 2)
explained_variance = (s6 ** 2)
explained_variance_ratio = explained_variance / total_variance

df_results = pd.DataFrame({
    'Explained Variance Ratio': explained_variance_ratio,
    'Cumulative Explained Variance': np.cumsum(explained_variance_ratio),
    'Singular Values': s6,
}, index=[f'PC{i+1}' for i in range(n_components)])
print("PCA summary:\n" + str(df_results))

# 6. Assign the 6 PC Scores back to each firm-year observation (row)
# Note: Full PC Scores in SVD space equal U * S
pc_scores = u6 * s6  

for i in range(n_components):
    df_merged[f'sim_pc{i+1}'] = pc_scores[:, i]
print("Sample of PC scores\n" + str(df_merged.head()))

# 7. Extract Column Loadings (Weight of each feature per Component)
df_loadings = pd.DataFrame(
    v6_T.T, 
    index=df_features.columns, 
    columns=[f'sim_pc{i+1}' for i in range(n_components)]
)
df_loadings['sim_pc1_abs'] = df_loadings['sim_pc1'].abs()
df_loadings.style.format("{:.3f}")
df_loadings.sort_values(by='sim_pc1_abs', ascending=False, inplace=True)
df_loadings.drop(columns=['sim_pc1_abs'], inplace=True)
print("Column Loadings (Weight of each feature per Component):\n" + str(df_loadings))

PCA summary:
     Explained Variance Ratio  Cumulative Explained Variance  Singular Values
PC1                  0.357887                       0.357887      3377.295244
PC2                  0.141825                       0.499712      2126.044936
PC3                  0.105472                       0.605184      1833.428584
PC4                  0.097592                       0.702776      1763.615775
PC5                  0.066986                       0.769763      1461.128745
PC6                  0.039064                       0.808827      1115.794593
Sample of PC scores
  registered_number  year  turnover  shareholders_funds  profit_loss_pretax  \
0          01840419  2006  0.083384            0.010006            0.007452   
1          01372811  2006 -0.007825           -0.014382           -0.000154   
2          02468057  2006 -0.009082           -0.015178            0.000022   
3          SC010677  2006  0.060551            0.026858            0.012361   
4          03221027  2006 

In [31]:
import pandas as pd

# Sum of squares of loadings per component
loadings_sum = df_loadings.apply(lambda x: np.sum(x**2), axis=0)
print("Sum of loadings per component:\n" + str(loadings_sum))

# For each component, take the squares of loadings, sort them, and then take the cumulative sum up to 0.95
# Ex: this could represent 6 or 7 features that explain 95% of the composition of the first component.
# Then, create a unique properties list of all the features that explaing each of the 6 components.
# Then, create a dataframe where rows are my unique property list, and the columns are the 6 components
# Values are loadings squared for each property in each component
# and 0 if the property is not in that component's top 95% of loadings squared.
feature_dict = {}
for i in range(n_components):
    comp_col = f'sim_pc{i+1}'
    loadings_squared = df_loadings[comp_col] ** 2
    loadings_squared_sorted = loadings_squared.sort_values(ascending=False)
    cumulative_sum = loadings_squared_sorted.cumsum()
    threshold_index = cumulative_sum[cumulative_sum <= 0.5].index
    feature_dict[comp_col] = threshold_index.tolist()
full_feature_list = [item for sublist in feature_dict.values() for item in sublist]
df_feature_matrix = pd.DataFrame({
    'property': full_feature_list
})
df_feature_matrix.set_index('property', inplace=True)
for i in range(n_components):
    should_include_arr = feature_dict[f'sim_pc{i+1}']
    comp_col = f'sim_pc{i+1}'
    loadings_squared = df_loadings[comp_col] ** 2
    loadings_filtered = loadings_squared[loadings_squared.index.isin(should_include_arr)]
    df_feature_matrix.loc[should_include_arr, comp_col] = loadings_filtered
pd.set_option('display.max_colwidth', None)
print("Feature matrix (rows: unique properties, columns: components, values: loadings squared):")
print(df_feature_matrix.to_string(sparsify=False))

Sum of loadings per component:
sim_pc1    1.0
sim_pc2    1.0
sim_pc3    1.0
sim_pc4    1.0
sim_pc5    1.0
sim_pc6    1.0
dtype: float64
Feature matrix (rows: unique properties, columns: components, values: loadings squared):
                               sim_pc1   sim_pc2   sim_pc3   sim_pc4   sim_pc5  sim_pc6
property                                                                               
shareholders_funds            0.096748       NaN       NaN       NaN       NaN      NaN
turnover                      0.089470       NaN       NaN       NaN       NaN      NaN
profit_loss_pretax            0.089001       NaN       NaN       NaN       NaN      NaN
ebitda                        0.088691       NaN       NaN       NaN       NaN      NaN
admin_expenses                0.087732       NaN       NaN       NaN       NaN      NaN
remuneration_employees             NaN  0.225024       NaN       NaN       NaN      NaN
wages                              NaN  0.224390       NaN       NaN   

In [ ]:
# Merge df_merged back into table_panel_name
table_panel_name = "working_yearly_with_tfp_wave"
table_panel = con.table(table_panel_name)
table_pcs = (
    ibis
    .memtable(df_merged)
    .select(['registered_number', 'year'] + [f'sim_pc{i+1}' for i in range(n_components)])
)
table_merged = (
    table_panel
    .left_join(table_pcs, ['registered_number', 'year'])
    .drop('registered_number_right', 'year_right')
)

# Print table_merged shape
print(f"Shape: {table_merged.count().execute()}x{len(table_merged.columns)}")
print(table_merged.limit(5).execute())

  registered_number  year  employees    fixed_total   total_assets  \
0          04144304  2018        268    1248.622674    6603.921920   
1          09015147  2023         87   23399.428095   61559.906332   
2          09593918  2019       1427    3734.869759    6743.484681   
3          09357457  2022       1551  405738.342553  449386.290426   
4          03769653  2016         48    1458.392219    9650.464924   

   average_wage          gva1          gva2  gva1_per_worker  gva2_per_worker  \
0     25.924319   8469.009282   9342.140216        31.600781        34.858732   
1     35.181676   6007.959476           NaN        69.057005              NaN   
2     18.643259  15156.386390  16636.820838        10.621154        11.658599   
3     23.813779  74089.368827  35933.198415        47.768774        23.167762   
4     39.865230   2964.962404   3038.928215        61.770050        63.311004   

   ...  tfp_wav1  tfp_wav2  tfp_wav3  nb_peers   sim_pc1   sim_pc2   sim_pc3  \
0  ...  2.81